# TC3 — Respostas às perguntas norteadoras

**Síntese executiva pronta para avaliação e apresentação.** A sequência começa pela capacidade preditiva (ROC e métricas), passa pela explicabilidade do modelo (SHAP e importância por permutação) e segue para as perguntas norteadoras sobre infraestrutura, INSE, continuidade municipal, finanças, clusters e metas. As conclusões usam os resultados já executados no notebook completo. Os gráficos são sínteses dos resultados existentes; não refazem o pipeline.

## 1. É possível estimar a alfabetização com os dados disponíveis?

**Resposta: sim, com capacidade preditiva moderada — não suficiente para tratar a previsão como diagnóstico definitivo.** O Random Forest final obteve ROC AUC **0,6705**, Average Precision **0,6841**, acurácia balanceada **0,6196** e F1 **0,6392** no teste independente. Há sinal preditivo, mas também erro relevante; o modelo deve apoiar priorização e investigação, não substituir indicadores oficiais.

*Referência no notebook completo: avaliação final independente, métricas, calibração e curvas ROC/Precisão-Recall.*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Identidade visual consistente: azuis e cinzas, sem cores de alto contraste desnecessárias.
AZUL_MARINHO = '#17324D'
AZUL_PETROLEO = '#267C8E'
AZUL_CLARO = '#8DBCC8'
CINZA = '#6B7280'
CINZA_CLARO = '#D9E1E8'
PALETA_ANOS = {2023: AZUL_MARINHO, 2024: AZUL_PETROLEO}

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.edgecolor': CINZA_CLARO, 'axes.labelcolor': AZUL_MARINHO,
    'xtick.color': CINZA, 'ytick.color': CINZA,
    'text.color': AZUL_MARINHO, 'axes.titleweight': 'bold',
    'axes.titlesize': 12, 'axes.labelsize': 10,
    'grid.color': CINZA_CLARO, 'grid.alpha': 0.8,
    'legend.frameon': False,
})

metricas = pd.DataFrame({
    'Métrica': ['ROC AUC', 'Precisão média (Average Precision)', 'Acurácia balanceada', 'F1'],
    'Valor': [0.670483, 0.684097, 0.619615, 0.639195]
}).sort_values('Valor')
fig, ax = plt.subplots(figsize=(9, 4.5))
barras = ax.barh(metricas['Métrica'], metricas['Valor'], color=AZUL_PETROLEO, height=0.62)
ax.set_xlim(0, 0.82)
ax.set_xlabel('Valor da métrica (escala de 0 a 1)')
ax.set_title('Desempenho do modelo no teste independente')
ax.grid(axis='x'); ax.set_axisbelow(True)
for barra, valor in zip(barras, metricas['Valor']):
    ax.text(valor + .012, barra.get_y() + barra.get_height()/2, f'{valor:.3f}', va='center', fontsize=9)
ax.spines[['top','right','left']].set_visible(False)
plt.tight_layout(); plt.show()
display(metricas)


## 2. Como o modelo se comporta? — curva ROC e métricas de avaliação

A avaliação final independente do Random Forest apresentou **ROC AUC = 0,6705**. O valor indica capacidade de distinguir as duas classes melhor do que uma classificação ao acaso, mas o desempenho ainda é **moderado** e não sustenta decisões individuais automatizadas.

| Métrica | Resultado | Leitura resumida |
|---|---:|---|
| ROC AUC | 0,6705 | Capacidade de discriminação moderada |
| Average Precision (AP) | 0,6841 | Qualidade da ordenação das previsões positivas, considerando a prevalência |
| Acurácia balanceada | 0,6196 | Desempenho médio entre as duas classes |
| F1-score | 0,6392 | Equilíbrio entre precisão e recall |
| Brier Score | 0,2273 | Erro das probabilidades previstas; menor é melhor |

**Como ler a curva ROC:** quanto mais próxima do canto superior esquerdo, melhor a discriminação. A linha diagonal representa desempenho equivalente ao acaso. A imagem abaixo foi produzida na execução final do pipeline; ela também inclui a curva precisão-recall e a matriz de confusão para manter o contexto da avaliação.

**Limite:** as métricas resumem o desempenho no conjunto de teste independente; não garantem que o modelo generalize igualmente para todos os estados, redes ou anos.

In [ ]:
from IPython.display import display, Image
import base64
display(Image(data=base64.b64decode('...')))

## 3. Quais variáveis mais influenciam as previsões? — leitura SHAP

A análise SHAP foi executada sobre uma **amostra de 100 registros** do conjunto de teste, usando o Random Forest final. O gráfico mostra a importância global pela média do valor absoluto de SHAP: valores maiores indicam maior contribuição média para as saídas do modelo nessa amostra.

**Principais sinais:** `sigla_uf` aparece com maior importância, seguida pelo valor de Bolsa Família transformado, INSE médio, receitas municipais transformadas e indicadores de infraestrutura. Isso reforça a relevância de fatores territoriais e contextuais para as previsões.

| Variável | Média do valor absoluto de SHAP |
|---|---:|

| Estado (UF) | 0.105783 |
| Valor do Bolsa Família (transformado) | 0.020112 |
| INSE médio | 0.012066 |
| Receitas municipais (transformadas) | 0.009798 |
| Acesso à internet | 0.007418 |
| Laboratório de informática | 0.007305 |
| Fundeb (transformado) | 0.006009 |
| Biblioteca/sala de leitura | 0.005671 |
| Rede de ensino | 0.004837 |
| Quadra de esportes | 0.004726 |
| PIB per capita (transformado) | 0.004294 |
| Energia elétrica pública | 0.003909 |
| Abastecimento de água | 0.003796 |
| Ano de referência | 0.003678 |
| Esgotamento sanitário | 0.003298 |
| Banda larga | 0.002039 |
| Indicador de ausência de cobertura financeira | 0.000000 |
| Indicador de ausência de cobertura INSE | 0.000000 |


**Importante:** SHAP explica como as variáveis contribuem para as previsões do modelo; não demonstra que elas causem alfabetização ou não alfabetização. A análise é global e baseada em amostra, portanto não deve ser interpretada como explicação individual definitiva. Os valores são da execução do pipeline registrada no notebook-fonte.

In [ ]:
from IPython.display import display, Image
import base64
display(Image(data=base64.b64decode('...')))


## 4. Quais fatores aparecem mais associados às estimativas?

**Objetivo:** identificar quais variáveis apresentam maior contribuição para as estimativas de alfabetização produzidas pelo modelo.

**Método:** utilizamos a importância por permutação para comparar a contribuição preditiva das variáveis selecionadas. Quanto maior a importância, maior a redução de desempenho observada quando a informação daquela variável é embaralhada.

**Resultados:** o estado (UF) apresenta a maior importância entre as variáveis exibidas, seguido por Bolsa Família, índice socioeconômico médio (INSE), rede de ensino e presença de laboratório de informática.

**Limitações:** a importância por permutação indica contribuição para o desempenho preditivo, mas não demonstra causalidade. Os resultados também dependem das variáveis disponíveis, da validação utilizada e da forma como os dados foram organizados.


In [ ]:
fatores = pd.DataFrame({
    'Variável': ['Estado (UF)', 'Bolsa Família*', 'Índice socioeconômico médio (INSE)', 'Rede de ensino', 'Escolas com laboratório de informática'],
    'Importância por permutação': [0.081454, 0.013114, 0.008716, 0.007759, 0.005250]
}).sort_values('Importância por permutação')
fig, ax = plt.subplots(figsize=(10, 4.8))
barras = ax.barh(fatores['Variável'], fatores['Importância por permutação'], color=[AZUL_CLARO, AZUL_CLARO, AZUL_CLARO, AZUL_PETROLEO, AZUL_MARINHO], height=.65)
ax.set_xlabel('Queda média na métrica ao permutar a variável')
ax.set_title('Estado (UF) apresenta a maior importância preditiva')
ax.grid(axis='x'); ax.set_axisbelow(True)
for barra, valor in zip(barras, fatores['Importância por permutação']):
    ax.text(valor + .001, barra.get_y() + barra.get_height()/2, f'{valor:.4f}', va='center', fontsize=9)
ax.spines[['top','right','left']].set_visible(False)
plt.tight_layout(); plt.show()
display(fatores)
print('* Bolsa Família foi transformado por signed_log1p; a importância não está expressa em reais.')


## 5. As condições de infraestrutura escolar estão associadas aos resultados de alfabetização?

**Objetivo:** comparar os resultados de alfabetização entre municípios nas faixas de menor e maior infraestrutura escolar.

**Método:** foram analisados oito indicadores de infraestrutura, comparando a taxa observada de alfabetização nas faixas extremas (0 e 4), para 2023 e 2024. Os gráficos apresentam os resultados observados, sem misturá-los às probabilidades estimadas pelo modelo.

**Resultados:** em diversos indicadores, a faixa de maior infraestrutura apresenta taxas observadas de alfabetização superiores às da menor faixa. Entretanto, a intensidade dessa diferença varia conforme o indicador e o ano, não sendo um padrão uniforme para todas as condições analisadas.

**Limitações:** os dados estão agregados por município. As diferenças observadas indicam associação, mas não demonstram causalidade. Além disso, indicadores com faixas ausentes não permitem comparação completa entre os extremos.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Nomes amigáveis dos indicadores
nomes_infra = {
    "in_biblioteca_sala_leitura": "Biblioteca / sala de leitura",
    "in_laboratorio_informatica": "Laboratório de informática",
    "in_quadra_esportes": "Quadra de esportes",
    "in_agua_rede_publica": "Água da rede pública",
    "in_energia_rede_publica": "Energia da rede pública",
    "in_esgoto_rede_publica": "Esgoto da rede pública",
    "in_internet": "Internet",
    "in_banda_larga": "Banda larga",
}

# Localizar e carregar o CSV de infraestrutura a partir da raiz do repositório.
from pathlib import Path

candidatos = [Path.cwd(), *Path.cwd().parents]
raiz_infra = next(
    (
        pasta for pasta in candidatos
        if (
            pasta / "resultados" / "tabelas"
            / "02_infraestrutura_por_faixa_ano.csv"
        ).is_file()
    ),
    None
)

if raiz_infra is None:
    raise FileNotFoundError(
        "Não encontrei resultados/tabelas/02_infraestrutura_por_faixa_ano.csv. "
        f"Diretório atual: {Path.cwd()}"
    )

infra = pd.read_csv(
    raiz_infra / "resultados" / "tabelas"
    / "02_infraestrutura_por_faixa_ano.csv"
)
print(f"CSV de infraestrutura carregado: {raiz_infra}")

# Preparar os dados sem modificar o DataFrame original
dados = infra.copy()

dados["faixa"] = pd.to_numeric(
    dados["faixa"], errors="coerce"
)
dados["ano_referencia"] = pd.to_numeric(
    dados["ano_referencia"], errors="coerce"
)
dados["taxa_observada"] = pd.to_numeric(
    dados["taxa_observada"], errors="coerce"
)

dados["Indicador"] = dados["feature"].map(nomes_infra)

# Selecionar somente a menor faixa (0) e a maior faixa (4)
dados = dados[dados["faixa"].isin([0, 4])].copy()

# Converter taxa para porcentagem
dados["Taxa observada (%)"] = dados["taxa_observada"] * 100

ordem = list(nomes_infra.values())

# Dois painéis: um para cada ano
fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 7),
    sharex=True,
    sharey=True
)

# Paleta monocromática azul, alinhada ao notebook
cores = {
    "Menor faixa": "#8CB9D9",  # Azul-claro
    "Maior faixa": "#2878B5",  # Azul principal
}
for ax, ano in zip(axes, [2023, 2024]):

    anual = dados[dados["ano_referencia"] == ano]

    # Ordenar indicadores pela lista definida
    indicadores_presentes = [
        indicador for indicador in ordem
        if indicador in anual["Indicador"].dropna().unique()
    ]

    y = np.arange(len(indicadores_presentes))
    altura = 0.34

    for deslocamento, faixa, rotulo in [
        (-altura / 2, 0, "Menor faixa (0)"),
        (altura / 2, 4, "Maior faixa (4)"),
    ]:
        valores = []

        for indicador in indicadores_presentes:
            linha = anual[
                (anual["Indicador"] == indicador)
                & (anual["faixa"] == faixa)
            ]

            valores.append(
                linha["Taxa observada (%)"].iloc[0]
                if not linha.empty
                else np.nan
            )

        ax.barh(
            y + deslocamento,
            valores,
            height=altura,
            color=cores[
                "Menor faixa" if faixa == 0 else "Maior faixa"
            ],
            label=rotulo
        )

        # Rótulos numéricos nas barras
        for pos, valor in zip(y + deslocamento, valores):
            if pd.notna(valor):
                ax.text(
                    valor + 0.25,
                    pos,
                    f"{valor:.1f}%",
                    va="center",
                    fontsize=8
                )

    ax.set_title(str(ano), fontsize=13, fontweight="bold")
    ax.set_yticks(y)
    ax.set_yticklabels(indicadores_presentes)
    ax.invert_yaxis()
    ax.set_xlabel("Taxa observada de alfabetização (%)")
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)
    ax.legend(
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.13),
        ncol=2,
        fontsize=9
    )
fig.suptitle(
    "Alfabetização observada por faixa de infraestrutura",
    fontsize=15,
    fontweight="bold"
)

fig.tight_layout(rect=[0, 0.08, 1, 0.96])
plt.show()


## 6. O INSE municipal está associado à alfabetização?

**Objetivo:** investigar a associação entre o Índice de Nível Socioeconômico (INSE) municipal e os resultados de alfabetização.

**Método:** foram analisadas as correlações do INSE com a taxa observada de alfabetização e com a probabilidade estimada pelo modelo, para 2023 e 2024.

**Resultados:** a correlação com a taxa observada foi baixa em ambos os anos (0,0586 em 2023 e 0,0240 em 2024). Já a correlação com a probabilidade estimada foi maior (0,2805 em 2023 e 0,1739 em 2024), indicando uma associação mais evidente com as estimativas do modelo do que com os resultados observados.

**Limitações:** o INSE utilizado corresponde à média municipal de 2023, empregada como referência contextual também na análise de 2024. Portanto, os resultados não representam o nível socioeconômico individual dos alunos e não demonstram causalidade.

In [ ]:
inse = pd.DataFrame({
    'Ano': [2023, 2024],
    'Correlação com taxa observada': [0.058576, 0.024030],
    'Correlação com probabilidade estimada': [0.280520, 0.173926]
})
fig, ax = plt.subplots(figsize=(8.5, 4.5))
x = range(len(inse)); w = 0.32
b1 = ax.bar([i-w/2 for i in x], inse['Correlação com taxa observada'], width=w, label='Taxa observada', color=AZUL_CLARO)
b2 = ax.bar([i+w/2 for i in x], inse['Correlação com probabilidade estimada'], width=w, label='Probabilidade estimada', color=AZUL_MARINHO)
ax.set_xticks(list(x), inse['Ano']); ax.set_ylabel('Correlação de Spearman')
ax.set_xlabel('Ano de referência')
ax.set_title('INSE municipal: associação com resultado observado e estimativa')
ax.axhline(0, color=CINZA, linewidth=.8); ax.grid(axis='y'); ax.set_axisbelow(True)
for bars in [b1,b2]: ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
ax.legend(loc='upper right'); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()
display(inse)


## 7. O desempenho municipal se mantém de 2023 para 2024?

**Objetivo:** avaliar se os resultados de alfabetização dos municípios apresentam continuidade entre 2023 e 2024.

**Método:** comparamos os resultados observados e as probabilidades estimadas pelo modelo nos dois anos, considerando os municípios com dados disponíveis para comparação.

**Resultados:** a análise permite investigar se municípios com melhores resultados em 2023 também tendem a apresentar melhores resultados em 2024. A comparação entre resultados observados e probabilidades estimadas deve ser interpretada separadamente, pois a estabilidade das previsões não significa necessariamente estabilidade dos resultados reais.

**Limitações:** a análise contempla apenas dois anos e depende da cobertura e da comparabilidade dos dados municipais. A associação entre os anos não demonstra causalidade nem garante que o padrão se mantenha no longo prazo.

In [ ]:
persist = pd.DataFrame({
    'Indicador comparado entre 2023 e 2024': ['Resultados observados', 'Probabilidades estimadas pelo modelo'],
    'Correlação de Spearman': [0.447454, 0.893588]
}).sort_values('Correlação de Spearman')
fig, ax = plt.subplots(figsize=(9, 3.8))
barras = ax.barh(persist['Indicador comparado entre 2023 e 2024'], persist['Correlação de Spearman'], color=[AZUL_CLARO, AZUL_MARINHO], height=.58)
ax.set_xlim(0, 1.04); ax.set_xlabel('Correlação de Spearman (0 a 1)')
ax.set_title('Persistência da ordenação municipal entre 2023 e 2024')
ax.grid(axis='x'); ax.set_axisbelow(True)
for barra, valor in zip(barras, persist['Correlação de Spearman']):
    ax.text(valor + .015, barra.get_y()+barra.get_height()/2, f'{valor:.3f}', va='center', fontsize=9)
ax.spines[['top','right','left']].set_visible(False)
plt.tight_layout(); plt.show()
display(persist)


## 8. Finanças públicas e condições socioeconômicas ajudam a explicar diferenças?

**Objetivo:** investigar a associação entre indicadores financeiros e socioeconômicos municipais e os resultados de alfabetização.

**Método:** foram analisadas variáveis relacionadas às finanças públicas e às condições socioeconômicas, comparando seus valores com os resultados observados e as estimativas do modelo.

**Resultados:** a análise permite identificar associações entre essas variáveis e as estimativas de alfabetização. A interpretação deve considerar que valores financeiros totais podem refletir diferenças no porte dos municípios, e não necessariamente maior investimento por aluno.

**Limitações:** as associações observadas não demonstram causalidade. Os resultados dependem da disponibilidade, da escala e da comparabilidade dos indicadores financeiros e socioeconômicos utilizados.

In [ ]:
from pathlib import Path

# Localiza o CSV já exportado pelo pipeline, sem refazer a análise.
candidatos = [Path.cwd(), *Path.cwd().parents]
raiz = next((p for p in candidatos if (p / 'resultados' / 'tabelas' / '07_financas_por_faixa_ano.csv').is_file()), None)
if raiz is None:
    raise FileNotFoundError("Não encontrei resultados/tabelas/07_financas_por_faixa_ano.csv. Abra o notebook a partir da raiz do repositório ou ajuste o caminho abaixo.")

financas = pd.read_csv(raiz / 'resultados' / 'tabelas' / '07_financas_por_faixa_ano.csv')
necessarias = {'feature', 'ano_referencia', 'faixa', 'taxa_observada'}
faltantes = necessarias - set(financas.columns)
if faltantes:
    raise ValueError(f"Colunas esperadas ausentes no CSV financeiro: {sorted(faltantes)}. Colunas disponíveis: {financas.columns.tolist()}")

# Mantém apenas as quatro dimensões financeiras/contextuais citadas na resposta.
mapa_features = {
    'valor_bolsa_familia__signed_log1p': 'Bolsa Família',
    'valor_fundeb__signed_log1p': 'Fundeb',
    'valor_receitas__signed_log1p': 'Receitas municipais',
    'valor_pib_per_capita__signed_log1p': 'PIB per capita',
}
financas['variavel'] = financas['feature'].map(mapa_features)
fin_plot = financas[financas['variavel'].notna()].copy()
if fin_plot.empty:
    print('Os nomes das variáveis no CSV diferem dos nomes previstos. Variáveis disponíveis:')
    display(financas[['feature']].drop_duplicates().sort_values('feature'))
    raise ValueError('Ajuste mapa_features aos nomes reais listados acima; nenhum valor foi inventado.')

# Ordena faixas de acordo com a ordem existente no CSV.
anos = sorted(fin_plot['ano_referencia'].dropna().unique())
fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), sharey=True)
for ax, (feature, nome) in zip(axes.flat, mapa_features.items()):
    d = fin_plot[fin_plot['feature'] == feature]
    if d.empty:
        ax.set_visible(False)
        continue
    for ano in anos:
        da = d[d['ano_referencia'] == ano]
        if not da.empty:
            ax.plot(da['faixa'].astype(str), da['taxa_observada'] * 100, marker='o', linewidth=2.3, markersize=5, color=PALETA_ANOS.get(int(ano), AZUL_PETROLEO), label=str(int(ano)))
    ax.set_title(f'{nome}: alfabetização por faixa')
    ax.set_ylabel('Taxa de alfabetização observada (%)')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y'); ax.set_axisbelow(True)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', alpha=.25)
    ax.legend(title='Ano')
fig.suptitle('Taxa de alfabetização observada por faixa de indicador financeiro/contextual', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
display(fin_plot[['variavel', 'ano_referencia', 'faixa', 'registros', 'feature_media', 'taxa_observada', 'probabilidade_media']].sort_values(['variavel', 'ano_referencia', 'faixa']))

## 9. É possível agrupar municípios em perfis distintos?

**Objetivo:** investigar se os municípios podem ser agrupados em perfis com características semelhantes, considerando as variáveis utilizadas na análise.

**Método:** foram comparadas soluções de agrupamento com 2 a 6 grupos, utilizando o coeficiente silhouette para avaliar a separação e a coesão dos grupos.

**Resultados:** a solução selecionada foi a de três grupos, com silhouette de aproximadamente 0,241. Esse resultado sugere separação limitada entre os perfis. Os grupos podem apoiar uma análise exploratória das diferenças entre municípios, mas devem ser interpretados com cautela.

**Limitações:** o coeficiente silhouette indica a qualidade da separação dos grupos, não sua relevância educacional por si só. A interpretação dos perfis depende das variáveis utilizadas, da distribuição dos registros e das características de cada grupo.

In [ ]:
sil = pd.DataFrame({'Número de grupos (k)': [2,3,4,5,6], 'Coeficiente silhouette': [0.236435,0.240664,0.239402,0.200605,0.196756]})
fig, ax = plt.subplots(figsize=(8.5, 4))
ax.plot(sil['Número de grupos (k)'], sil['Coeficiente silhouette'], marker='o', linewidth=2.5, markersize=7, color=AZUL_PETROLEO)
ax.set_xticks(sil['Número de grupos (k)']); ax.set_ylabel('Coeficiente silhouette')
ax.set_xlabel('Quantidade de grupos (k)')
ax.set_title('Comparação das alternativas de agrupamento')
ax.grid(axis='y'); ax.set_axisbelow(True)
for x, y in zip(sil['Número de grupos (k)'], sil['Coeficiente silhouette']): ax.annotate(f'{y:.3f}', (x,y), xytext=(0,8), textcoords='offset points', ha='center', fontsize=8)
ax.annotate('Solução selecionada: 3 grupos', (3,0.240664), xytext=(3.45,0.225), arrowprops={'arrowstyle':'->','color':AZUL_MARINHO}, fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()
display(sil)
display(pd.DataFrame({'Grupo': [0,1,2], 'Registros município-ano': [5058,7255,25], 'Interpretação': ['Perfil amplo', 'Perfil amplo', 'Grupo muito pequeno; interpretar com cautela']}))


## 10. Quais municípios podem não alcançar as metas futuras?

**Objetivo:** examinar a disponibilidade de metas municipais para a análise de risco de alfabetização.

**Método:** comparamos a quantidade de municípios com e sem meta correspondente nos anos de referência de 2023 e 2024.

**Resultados:** em 2023, 4.664 municípios possuem meta correspondente e 145 não possuem. Em 2024, 5.327 possuem meta correspondente e 99 não possuem.

**Limitação importante:** este gráfico apresenta a **cobertura das metas**, não identifica quais municípios estão em risco de não alcançá-las. Para responder diretamente à pergunta, é necessário cruzar as estimativas de alfabetização com a meta correspondente de cada município. A ausência de uma meta também não significa, por si só, maior risco.


In [ ]:
cobertura = pd.DataFrame({
    'Ano de referência': [2023, 2023, 2024, 2024],
    'Cobertura da meta': ['Com meta correspondente', 'Sem meta correspondente', 'Com meta correspondente', 'Sem meta correspondente'],
    'Municípios': [4664, 145, 5327, 99]
})
fig, ax = plt.subplots(figsize=(9, 4.5))
categorias = ['Com meta correspondente', 'Sem meta correspondente']
x = range(len(categorias)); largura = 0.32
for deslocamento, ano in [(-largura/2, 2023), (largura/2, 2024)]:
    d = cobertura[cobertura['Ano de referência'] == ano].set_index('Cobertura da meta').reindex(categorias)
    barras = ax.bar([i + deslocamento for i in x], d['Municípios'], largura, label=str(ano), color=PALETA_ANOS[ano])
    ax.bar_label(barras, padding=3, fmt='%.0f', fontsize=9)
ax.set_xticks(list(x), ['Com meta', 'Sem meta'])
ax.set_ylabel('Quantidade de municípios')
ax.set_xlabel('Disponibilidade de meta municipal correspondente')
ax.set_title('Cobertura das metas municipais por ano')
ax.legend(title='Ano', ncol=2)
ax.set_ylim(0, 6000); ax.grid(axis='y'); ax.set_axisbelow(True)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()
display(cobertura)


### Convenções de leitura dos gráficos

Os rótulos foram simplificados para facilitar a leitura, preservando os nomes técnicos no código e nas tabelas de origem. Variáveis financeiras transformadas aparecem com nomes amigáveis no gráfico, mas não devem ser interpretadas diretamente como valores em reais. Os gráficos financeiros são carregados do CSV exportado pelo pipeline; os demais gráficos resumem métricas e resultados já calculados no notebook completo.


## Conclusão executiva

1. **Predição:** possível, com desempenho moderado (ROC AUC 0,670).
2. **Fatores:** UF domina a importância; Bolsa Família, INSE, rede e laboratório aparecem em seguida. Importância não é causalidade.
3. **Infraestrutura:** as taxas observadas tendem a ser maiores nas faixas mais altas de biblioteca/sala de leitura, com variação entre anos e faixas.
4. **INSE:** correlação fraca com a taxa observada; moderada com a probabilidade estimada em 2023 e menor em 2024.
5. **Histórico municipal:** resultados observados têm persistência moderada (rho 0,447); previsões, persistência alta (rho 0,894).
6. **Finanças:** oferecem contexto, mas os dados não isolam efeito causal nem gasto por aluno.
7. **Clusters:** três grupos são uma solução exploratória, com separação fraca e um grupo muito pequeno.
8. **Metas futuras:** este notebook mostra a cobertura das metas; não identifica diretamente os municípios em risco. Essa identificação exige comparar a estimativa de cada município com sua meta correspondente.

**Limite transversal:** os dados são agregados/contextuais em várias análises; não sustentam conclusões individuais sobre alunos nem afirmações causais.